# System validation notebook: Digital Restoration of Indian Folk Paintings
Reads the logs and result files written by the pipeline and reports: **patch-conversion rates**, **augmentation (synthetic damage) statistics**, **comparative loss curves**, and the **final metrics** for all four milestones. Nothing here is typed in by hand: every number is read from `data/`, `runs/` and `results/`. A section whose files are missing says so and is skipped.

Run it from the working folder (`/kaggle/working/folk-restore`) after the pipeline has run.

In [ ]:
import os, glob, json, csv
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
if os.path.exists("/kaggle/working/folk-restore"):
    os.chdir("/kaggle/working/folk-restore")
import sys; sys.path.insert(0, os.getcwd())
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3})
def have(p):
    ok = os.path.exists(p)
    if not ok: print(f"[skipped] {p} not found")
    return ok
print("working dir:", os.getcwd())

## 1. Dataset and patch conversion
How many source paintings survive filtering and de-duplication, and how many 512x512 tiles each one yields. The split is **by image**, so no painting contributes tiles to both training and test.

In [ ]:
if have("data/manifest.csv"):
    man = pd.read_csv("data/manifest.csv", dtype={"image_id": str})
    raw = [f for f in glob.glob("data/raw/**/*", recursive=True) if f.lower().endswith((".jpg",".jpeg",".png",".webp",".bmp",".tif",".tiff"))]
    n_img = man.image_id.nunique()
    per_img = man.groupby("image_id").size()
    print(f"raw image files found in data/raw : {len(raw)}")
    print(f"kept after size/duplicate filters : {n_img}")
    print(f"tiles produced                    : {len(man)}  -> {len(man)/n_img:.1f} tiles per image on average "
          f"(min {per_img.min()}, median {int(per_img.median())}, max {per_img.max()})")
    tab = man.pivot_table(index="style", columns="split", values="tile_id", aggfunc="count", fill_value=0)
    tab_img = man.drop_duplicates("image_id").pivot_table(index="style", columns="split", values="image_id", aggfunc="count", fill_value=0)
    print("\ntiles per style x split"); display(tab)
    print("images per style x split"); display(tab_img)
    leak = set(man[man.split=="train"].image_id) & set(man[man.split.isin(["val","test"])].image_id)
    print("images appearing in both train and val/test (must be 0):", len(leak))
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    tab.plot.bar(ax=ax[0], stacked=True, title="tiles per style and split"); ax[0].set_xlabel("")
    per_img.plot.hist(bins=20, ax=ax[1], title="tiles per source image"); ax[1].set_xlabel("tiles")
    plt.tight_layout(); plt.show()

## 2. Synthetic damage engine: augmentation metrics
Training damage is generated on the fly, so here we draw a sample from the engine and measure what it produces: how often each defect appears, how much of a tile it covers, how strong the fading is, and how often unlabelled design-line decoys appear (the hard negatives added to stop the segmenter erasing real line work).

In [ ]:
if have("data/manifest.csv"):
    from degrade import degrade
    from skimage.color import deltaE_ciede2000, rgb2lab
    N = 300
    rng = np.random.default_rng(0)
    train = man[man.split == "train"].sample(min(N, (man.split=="train").sum()), random_state=0)
    rows, ex = [], []
    for k, r in enumerate(train.itertuples()):
        clean = cv2.cvtColor(cv2.imread("data/" + r.path), cv2.COLOR_BGR2RGB)[:256, :256]
        d = degrade(clean, rng, fade_prob=0.6, decoy_prob=0.4)
        row = {f"cover_{c}": float(m.mean()) for c, m in d["masks"].items()}
        row["cover_union"] = float(np.any([m > 0 for m in d["masks"].values()], axis=0).mean())
        row["faded"] = bool(d["faded"])
        row["decoy"] = bool((d["clean"] != clean).any())
        row["fade_dE00"] = float(deltaE_ciede2000(rgb2lab(d["faded_only"]), rgb2lab(d["clean"])).mean()) if d["faded"] else 0.0
        rows.append(row)
        if k < 4: ex.append((d["clean"], d["image"], np.stack([d["masks"][c] for c in ("crack","loss","stain")], -1) * 255))
    aug = pd.DataFrame(rows)
    summ = pd.DataFrame({
        "tiles containing defect (%)": [100*(aug[f"cover_{c}"] > 0).mean() for c in ("crack","loss","stain")],
        "mean coverage when present (%)": [100*aug.loc[aug[f"cover_{c}"] > 0, f"cover_{c}"].mean() for c in ("crack","loss","stain")],
        "p95 coverage (%)": [100*aug[f"cover_{c}"].quantile(.95) for c in ("crack","loss","stain")],
    }, index=["crack","loss","stain"]).round(2)
    display(summ)
    print(f"mean total damaged area per tile: {100*aug.cover_union.mean():.1f}%   |   tiles with design-line decoys: {100*aug.decoy.mean():.0f}%   |   "
          f"faded tiles: {100*aug.faded.mean():.0f}%  (mean dE00 of the fade on those: {aug.loc[aug.faded,'fade_dE00'].mean():.2f})")
    fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
    aug[["cover_crack","cover_loss","cover_stain"]].replace(0, np.nan).plot.hist(bins=30, alpha=.6, ax=ax[0], title="coverage per tile (when present)")
    aug.loc[aug.faded, "fade_dE00"].plot.hist(bins=25, ax=ax[1], title="fade strength (dE00)")
    aug.cover_union.plot.hist(bins=30, ax=ax[2], title="total damaged fraction")
    plt.tight_layout(); plt.show()
    fig, ax = plt.subplots(3, len(ex), figsize=(3*len(ex), 9), squeeze=False)
    for j, (c, dmg, m) in enumerate(ex):
        for i, (im, t) in enumerate(((c, "clean (+decoys)"), (dmg, "damaged + faded"), (m, "masks R=crack G=loss B=stain"))):
            ax[i, j].imshow(im); ax[i, j].set_title(t, fontsize=9); ax[i, j].axis("off")
    plt.tight_layout(); plt.show()

## 3. Comparative loss curves across epochs

In [ ]:
# Milestone 1: segmenter variants
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
runs = {"unet_rich": "U-Net, CLAHE + black-hat (main)", "unet_v2": "U-Net + design-line decoys", "unet_rgb": "U-Net, RGB only (ablation)"}
for r, lab in runs.items():
    p = f"runs/{r}/log.csv"
    if os.path.exists(p):
        df = pd.read_csv(p); ax[0].plot(df.epoch, df.train_loss, marker="o", ms=3, label=lab); ax[1].plot(df.epoch, df.dice_union, marker="o", ms=3, label=lab)
ax[0].set_title("M1 segmenter: training loss (BCE+Dice)"); ax[1].set_title("M1 segmenter: validation union Dice (higher = better)")
for a_ in ax: a_.set_xlabel("epoch"); a_.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Milestone 2: LaMa fine-tuning
if have("runs/lama_ft/log.csv"):
    df = pd.read_csv("runs/lama_ft/log.csv")
    fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
    d1 = df[df.epoch > 0]
    ax[0].plot(d1.epoch, d1.l1, marker="o", ms=3); ax[0].set_title("M2 masked L1 (train)")
    ax[1].plot(d1.epoch, d1.adv_g, marker="o", ms=3, label="generator adv."); ax[1].plot(d1.epoch, d1.d_loss, marker="o", ms=3, label="critic hinge"); ax[1].legend(); ax[1].set_title("M2 adversarial terms")
    ax[2].plot(df.epoch, df.val_psnr_region, marker="o", ms=3); ax[2].axhline(df.val_psnr_region.iloc[0], ls="--", c="gray", label="zero-shot LaMa (epoch 0)"); ax[2].legend()
    ax[2].set_title("M2 validation PSNR on repaired region (dB)")
    for a_ in ax: a_.set_xlabel("epoch")
    plt.tight_layout(); plt.show()

In [ ]:
# Milestone 3: de-fader, L1-only (step 8) vs Pix2Pix variants
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
if os.path.exists("runs/defade/log.csv"):
    df = pd.read_csv("runs/defade/log.csv"); ax[0].plot(df.epoch, df.train_loss, marker="o", ms=3, label="step 8, L1 only (train loss)"); ax[1].plot(df.epoch, df.val_de00_defaded, marker="o", ms=3, label="step 8: L1 U-Net, no reference")
    ax[1].axhline(df.val_de00_faded.iloc[0], ls="--", c="gray", label="faded input")
for name, lab in (("defade_gan", "Pix2Pix (L1 + GAN)"), ("defade_l1ref", "same model, GAN off (ablation)")):
    p = f"runs/{name}/log.csv"
    if os.path.exists(p):
        df = pd.read_csv(p); ax[0].plot(df.epoch, df.l1, marker="o", ms=3, label=f"{lab}: L1 (train)")
        ax[1].plot(df.epoch, df.val_de00_noref, marker="o", ms=3, label=f"{lab}, no margin"); ax[1].plot(df.epoch, df.val_de00_ref, marker="s", ms=3, label=f"{lab}, with margin")
ax[0].set_title("M3 training L1 (different scales: compare shape, not height)"); ax[1].set_title("M3 validation CIEDE2000 (lower = better)")
for a_ in ax: a_.set_xlabel("epoch"); a_.legend(fontsize=7)
plt.tight_layout(); plt.show()

## 4. Final metrics on the held-out synthetic test set
Synthetic-test numbers measure handling of *simulated* damage. Real paintings have no ground truth, so those results are qualitative (section 5).

In [ ]:
for title, p in (("M1: segmentation (Dice)", "results/seg_metrics.json"),
                 ("M1 (v2, with design-line decoys): segmentation", "results_v2/seg_metrics.json")):
    if os.path.exists(p): print(f"### {title}"); print(json.dumps(json.load(open(p)), indent=2))
for title, p in (("M2: inpainting, Telea vs pretrained LaMa (step 4)", "results/summary.md"),
                 ("M2: inpainting, adding fine-tuned LaMa (step 10)", "results/inpaint_ft_summary.md"),
                 ("M3: global Reinhard baseline", "results/defade_metrics.md"),
                 ("M3: L1-only learned de-fader (step 8)", "results/defade_learned.md"),
                 ("M3: Pix2Pix de-fader with margin references (step 11)", "results/defade_gan.md"),
                 ("M3 ablation: same model without the GAN loss", "results_l1ref/defade_gan.md"),
                 ("M4: does the uncertainty map predict real error?", "results/uncertainty_eval.md"),
                 ("M4: super-resolution and tiling", "results/sr_eval.md")):
    if os.path.exists(p): display(Markdown(f"### {title}\n\n" + open(p).read()))
    else: print(f"[skipped] {p}")

## 5. Real paintings (qualitative)
The review image tints low-confidence areas red; the confidence matrix is saved next to each restored image.

In [ ]:
files = sorted(glob.glob("results/final/*_review.png"))[:4]
if not files: print("[skipped] results/final/*_review.png (run restore.py on data/real_damaged)")
for f in files:
    stem = f.replace("_review.png", "")
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    for a_, (suffix, t) in zip(ax, (("_restored.png", "restored"), ("_confidence.png", "confidence (red = speculative)"), ("_review.png", "review overlay"))):
        a_.imshow(cv2.cvtColor(cv2.imread(stem + suffix), cv2.COLOR_BGR2RGB)); a_.set_title(os.path.basename(stem) + " - " + t, fontsize=9); a_.axis("off")
    plt.tight_layout(); plt.show()
    c = np.load(stem + "_confidence.npy"); print(f"{os.path.basename(stem)}: confidence matrix {c.shape}, mean {c.mean():.3f}, share below 0.5: {100*(c<0.5).mean():.1f}%")